# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:15<00:00,  5.27s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

'Title: Refurb Dell Latitude Laptop Sale at Dell Refurbished: Extra 40% off + shipping varies\nDetails: Dell Refurbished is offering a range of refurbished Latitude laptops, with prices starting at $215 after promo code "DELLSUMMER40" is applied. The extra 40% off coupon gets some of the lowest prices we\'ve seen this year. Some exclusions apply like Hot Deals. Each purchase includes the same limited hardware warranty Dell offers on new systems. Sale ends June 14, 2026. Shop Now at Dell Refurbished Store\nFeatures: Grades A and B cosmetic conditions available Processors ranging from 10th to 13th Gen Intel Core RAM options: 8GB, 16GB, 32GB, or 64GB Storage: 256GB, 512GB, or 1TB+ SSD Displays from 13.3" to 17" FHD, QHD+, or UHD+ Windows 10 Pro, Windows 11 Pro, or no OS options\nURL: https://www.dealnews.com/Refurb-Dell-Latitude-Laptop-Sale-at-Dell-Refurbished-Extra-40-off-shipping-varies/21839445.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: ESR 25W 3-in-1 Wireless Charger w/ MagSafe for $26 + free shipping
Details: Apply promo code "ESR2C571US85" to drop this ESR 3-in-1 MagSafe charger to $26, down from its regular price of $140. That beats the Amazon price today by almsot $20. The charger includes Apple-certified 15W MagSafe charging for iPhone, 5W fast charging for Apple Watch, and a built-in cooling fan that ke

In [8]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='ESR 3‑in‑1 MagSafe wireless charging pad that supports Apple‑certified 15W MagSafe charging for iPhone 12 and later, a 5W Apple Watch charging puck, and a dedicated spot for AirPods. It delivers up to 25W total output, uses a CryoBoost cooling fan to keep phone temperature below 98°F during charging, and offers a Dark Charging Mode that disables the fan while maintaining 15W. The unit ships with a 33W USB‑C power adapter and a 5‑ft USB‑C cable.', price=26.0, url='https://www.dealnews.com/products/ESR/ESR-25-W-3-in-1-Wireless-Charger-w-Mag-Safe/499109.html?iref=rss-c142'), Deal(product_description='EcoFlow RAPID Pro portable power bank with a 10,000 mAh capacity and three‑in‑one functionality for charging phones, tablets, and other USB devices. It’s a compact, travel‑friendly power bank designed for fast recharging of mobile devices while on the go and includes multiple output options for versatile device compatibility.', price=65.0, url='h

In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


ESR 3‑in‑1 MagSafe wireless charging pad that supports Apple‑certified 15W MagSafe charging for iPhone 12 and later, a 5W Apple Watch charging puck, and a dedicated spot for AirPods. It delivers up to 25W total output, uses a CryoBoost cooling fan to keep phone temperature below 98°F during charging, and offers a Dark Charging Mode that disables the fan while maintaining 15W. The unit ships with a 33W USB‑C power adapter and a 5‑ft USB‑C cable.
26.0
https://www.dealnews.com/products/ESR/ESR-25-W-3-in-1-Wireless-Charger-w-Mag-Safe/499109.html?iref=rss-c142

EcoFlow RAPID Pro portable power bank with a 10,000 mAh capacity and three‑in‑one functionality for charging phones, tablets, and other USB devices. It’s a compact, travel‑friendly power bank designed for fast recharging of mobile devices while on the go and includes multiple output options for versatile device compatibility.
65.0
https://www.dealnews.com/Eco-Flow-RAPID-Pro-10-000-m-Ah-3-in-1-Power-Bank-for-65-free-shipping/21839386.

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.scanner_agent import ScannerAgent

In [ ]:
agent = ScannerAgent()
result = agent.scan()

In [ ]:
result

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [ ]:
load_dotenv(override=True)

In [ ]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("MASSIVE DEAL!!")

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")